# RSNA Knee Abnormality Detection - 10-Model Live Inference Ensemble (V24 + V29)

**Pure Inference Pipeline (~15-20 Minutes Runtime):**
- **Zero Training**: Skips all training to conserve GPU quota and eliminate timeout risk.
- **10 Pre-Trained Models**: Blends 5 Folds of V24 (Seed 42, 0.657 LB) + 5 Folds of V29 (Seed 2026, 0.651 LB).
- **Dual-Seed Variance Reduction**: Reconciles split-specific biases on unseen test studies.
- **Inference**: 4-Way Test-Time Augmentation (TTA: Original, H-Flip, Rot90, Rot270) on live hidden test DICOMs.
- **Baker's Post-Processing**: Exact empirical prior (0.010) applied to Baker's cyst channel.

In [ ]:
import os, re, gc, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import pydicom
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import timm
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
print(f'PyTorch {torch.__version__} | CUDA {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

try:
    import cv2
    HAS_CV2 = True
    print('CLAHE: using OpenCV')
except ImportError:
    HAS_CV2 = False
    from PIL import ImageOps
    print('CLAHE: using PIL fallback')

## 1. Configuration & Model Weight Discovery

In [ ]:
class CFG:
    backbone       = 'efficientnet_b2'
    sag_slices     = 8
    cor_slices     = 6
    axi_slices     = 6
    total_slices   = 20
    image_size     = (256, 256)
    num_classes    = 12
    batch_size     = 8           # Larger batch size for fast inference
    use_amp        = True
    bakers_idx     = 9
    bakers_prior   = 0.010
    device         = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    target_cols    = [
        'ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus',
        'Medial OA', 'Lateral OA', 'PF OA', 'Effusion',
        'Synovitis', "Baker's", 'Contusion', 'Fracture'
    ]

# Auto-discover all available trained checkpoints (V24 and V29)
all_checkpoints = []
search_dirs = [
    Path('/kaggle/input'),
    Path('.'),
]

for sdir in search_dirs:
    if sdir.exists():
        for p in sdir.rglob('*.pth'):
            p_str = str(p).lower()
            # Check if it's a trained model checkpoint (fold0..4)
            if 'fold' in p_str and ('model' in p_str or 'b2' in p_str):
                if p not in all_checkpoints:
                    all_checkpoints.append(p)

print(f'Total trained fold checkpoints discovered: {len(all_checkpoints)}')
for cp in sorted(all_checkpoints):
    print(f'  - {cp}')

## 2. Fast DICOM Reader with CLAHE & Asymmetric Slicing

In [ ]:
PLANE_KEYWORDS = {
    'sagittal': ['sag'],
    'coronal':  ['cor'],
    'axial':    ['axi', 'tra'],
}

def apply_clahe(arr_uint8):
    if HAS_CV2:
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        return clahe.apply(arr_uint8)
    else:
        img = Image.fromarray(arr_uint8, mode='L')
        return np.array(ImageOps.equalize(img))

def read_dcm(path):
    try:
        dcm = pydicom.dcmread(str(path))
        arr = dcm.pixel_array.astype(np.float32)
        arr = arr * float(getattr(dcm, 'RescaleSlope', 1)) + float(getattr(dcm, 'RescaleIntercept', 0))
        lo, hi = np.percentile(arr, 1), np.percentile(arr, 99)
        arr = np.clip(arr, lo, hi)
        if hi > lo: arr = (arr - lo) / (hi - lo) * 255.0
        arr = arr.astype(np.uint8)
        arr = apply_clahe(arr)
        return arr
    except Exception:
        return np.zeros(CFG.image_size, dtype=np.uint8)

def sample_slices(files, n):
    if not files:
        return [np.zeros(CFG.image_size, dtype=np.uint8)] * n
    idx = (np.linspace(0, len(files)-1, n, dtype=int)
           if len(files) >= n
           else list(range(len(files))) + [len(files)-1] * (n - len(files)))
    return [read_dcm(files[i]) for i in idx]

class KneeAdaptive3ViewDataset(Dataset):
    def __init__(self, df, series_df, images_dir, transform):
        self.df        = df.reset_index(drop=True)
        self.series_df = series_df
        self.img_dir   = Path(images_dir)
        self.tf        = transform

    def __len__(self): return len(self.df)

    def _plane_files(self, study_id, keywords):
        files = []
        if not self.series_df.empty:
            for kw in keywords:
                mask = (
                    (self.series_df['StudyInstanceUID'] == study_id) &
                    (self.series_df['Anatomical_Plane'].astype(str).str.lower().str.contains(kw))
                )
                for _, row in self.series_df[mask].iterrows():
                    d = self.img_dir / str(study_id) / str(row['SeriesInstanceUID'])
                    if d.exists(): files.extend(sorted(d.glob('*.dcm')))
        return files

    def _all_files(self, study_id):
        d = self.img_dir / str(study_id)
        return sorted(d.rglob('*.dcm')) if d.exists() else []

    def _to_tensor(self, arr_list):
        return torch.stack([self.tf(Image.fromarray(a).convert('RGB')) for a in arr_list])

    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        sid  = row['StudyInstanceUID']
        sag_f = self._plane_files(sid, PLANE_KEYWORDS['sagittal'])
        cor_f = self._plane_files(sid, PLANE_KEYWORDS['coronal'])
        axi_f = self._plane_files(sid, PLANE_KEYWORDS['axial'])
        if sum([bool(sag_f), bool(cor_f), bool(axi_f)]) >= 2:
            all_f  = self._all_files(sid)
            slices = (sample_slices(sag_f or all_f, CFG.sag_slices) +
                      sample_slices(cor_f or all_f, CFG.cor_slices) +
                      sample_slices(axi_f or all_f, CFG.axi_slices))
        else:
            slices = sample_slices(self._all_files(sid), CFG.total_slices)
        images = self._to_tensor(slices)
        return {'images': images, 'study_id': str(sid)}

val_transform = T.Compose([
    T.Resize(CFG.image_size),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
print('Dataset transforms configured successfully.')

## 3. Architecture & Test-Time Augmentation (TTA) Inference

In [ ]:
class AttentionPool(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.attn = nn.Sequential(nn.Linear(dim, 128), nn.Tanh(), nn.Linear(128, 1))

    def forward(self, x):
        w = torch.softmax(self.attn(x), dim=1)
        return (x * w).sum(dim=1)

class KneeModel(nn.Module):
    def __init__(self, backbone_name='efficientnet_b2'):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, pretrained=False, num_classes=0)
        D = self.backbone.num_features # 1408 for B2
        self.pool = AttentionPool(D)
        self.head = nn.Sequential(
            nn.Dropout(0.4),
            nn.Linear(D, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Dropout(0.25),
            nn.Linear(512, CFG.num_classes),
        )

    def forward(self, images):
        B, S, C, H, W = images.shape
        feats = self.backbone(images.view(B*S, C, H, W))
        feats = feats.view(B, S, -1)
        return self.head(self.pool(feats))

@torch.no_grad()
def tta_infer(model, loader):
    model.eval()
    all_p = []
    for batch in tqdm(loader, desc='  predicting'):
        imgs = batch['images'].to(CFG.device, non_blocking=True)
        with torch.cuda.amp.autocast(enabled=CFG.use_amp):
            # 4-way TTA: Original, H-Flip, Rot90, Rot270
            p0 = torch.sigmoid(model(imgs))
            p1 = torch.sigmoid(model(torch.flip(imgs, dims=[-1])))
            p2 = torch.sigmoid(model(torch.rot90(imgs, k=1, dims=[-2,-1])))
            p3 = torch.sigmoid(model(torch.rot90(imgs, k=3, dims=[-2,-1])))
            p_avg = (p0 + p1 + p2 + p3) / 4.0
        all_p.append(p_avg.cpu().numpy())
    return np.concatenate(all_p)

print('Model architecture and TTA functions defined.')

## 4. Live Test Set Inference & 10-Model Blending

In [ ]:
DATA_DIR = None
for p in [
    Path('/kaggle/input/rsna-knee-abnormality-detection'),
    Path('/kaggle/input/competitions/rsna-knee-abnormality-detection'),
    Path('./data'),
]:
    if p.exists(): DATA_DIR = p; break
if DATA_DIR is None: DATA_DIR = Path('.')

print('Loading test data...')
test_csv     = DATA_DIR / 'test.csv'
sample_sub   = DATA_DIR / 'sample_submission.csv'
test_ser_csv = DATA_DIR / 'test_series.csv'

test_df     = pd.read_csv(test_csv if test_csv.exists() else sample_sub)
test_series = pd.read_csv(test_ser_csv) if test_ser_csv.exists() else pd.DataFrame()
print(f'Test studies to evaluate: {len(test_df)}')

test_ds = KneeAdaptive3ViewDataset(test_df, test_series, DATA_DIR/'test_series', val_transform)
test_ld = DataLoader(test_ds, batch_size=CFG.batch_size, shuffle=False,
                     num_workers=2, pin_memory=True)

if len(all_checkpoints) == 0:
    print('WARNING: No pre-trained checkpoints found! Using uniform prior.')
    final = np.full((len(test_df), CFG.num_classes), 0.05, dtype=np.float32)
else:
    print(f'Running live inference across all {len(all_checkpoints)} models...')
    model_preds = []
    for i, cp_path in enumerate(all_checkpoints):
        print(f'[{i+1}/{len(all_checkpoints)}] Loading {cp_path.name}...')
        m = KneeModel(CFG.backbone).to(CFG.device)
        state_dict = torch.load(str(cp_path), map_location=CFG.device)
        m.load_state_dict(state_dict, strict=True)
        preds = tta_infer(m, test_ld)
        model_preds.append(preds)
        del m, state_dict; gc.collect(); torch.cuda.empty_cache()
    
    final = np.mean(model_preds, axis=0)

# Numerical safety clipping & Baker's cyst empirical prior
final = np.clip(np.nan_to_num(final, nan=0.05), 0.001, 0.999)
final[:, CFG.bakers_idx] = CFG.bakers_prior

# Build and save official competition submission
sub = pd.DataFrame(final, columns=CFG.target_cols)
sub.insert(0, 'StudyInstanceUID', test_df['StudyInstanceUID'].values)
sub.to_csv('submission.csv', index=False)

print(f'\nSubmission successfully generated: {sub.shape}')
print(f'Any missing values: {sub.isnull().any().any()}')
print('Head of submission:')
print(sub.head(3))
assert os.path.exists('submission.csv')
print('\nDone! 10-Model Live Ensemble completed successfully in ~15 minutes.')